# 01 - Exploración de Datos

Exploración de los datasets públicos Spider y BIRD, y conversión a MySQL + español.

In [ ]:
import json
import os
import re
from pathlib import Path
import pandas as pd

## 1. Dataset Spider

Spider contiene ~8,000 pares pregunta-SQL sobre 200 bases de datos. Las queries están en SQLite.

In [ ]:
# Descargar Spider si no está presente
spider_dir = Path('datos/spider_raw')
spider_dir.mkdir(parents=True, exist_ok=True)

# URL de descarga
print('Spider: https://yale-lily.github.io/spider')
print('Descargar train_spider.json y tables.json')
print('Colocar en datos/spider_raw/')

In [ ]:
# Cargar Spider
def cargar_spider(ruta_train='datos/spider_raw/train_spider.json',
                  ruta_tables='datos/spider_raw/tables.json'):
    with open(ruta_train, 'r') as f:
        train_data = json.load(f)
    with open(ruta_tables, 'r') as f:
        tables_data = json.load(f)
    
    print(f'Spider train: {len(train_data)} ejemplos')
    print(f'Spider tablas: {len(tables_data)} bases de datos')
    
    return train_data, tables_data

## 2. Conversión SQLite → MySQL

Principales diferencias a convertir:
- `AUTOINCREMENT` → `AUTO_INCREMENT`
- Tipos de datos (TEXT → VARCHAR, etc.)
- Funciones de fecha
- `LIMIT` sintaxis compatible
- Agregar COMMENTs

In [ ]:
def sqlite_a_mysql(sqlite_sql):
    """Convierte una query SQLite a MySQL."""
    sql = sqlite_sql
    
    # Reemplazos comunes
    reemplazos = {
        'LIMIT 1 OFFSET': 'LIMIT 1 OFFSET',
        'INTERSECT': 'INTERSECT',
        'EXCEPT': 'EXCEPT',  # MySQL 8.0+ soporta EXCEPT
    }
    
    # Convertir funciones de fecha
    sql = re.sub(r"STRFTIME\('%Y',\s*(.+?)\)", r'YEAR(\1)', sql, flags=re.IGNORECASE)
    sql = re.sub(r"STRFTIME\('%m',\s*(.+?)\)", r'MONTH(\1)', sql, flags=re.IGNORECASE)
    
    # Convertir IFNULL
    sql = re.sub(r'IFNULL\(', 'COALESCE(', sql, flags=re.IGNORECASE)
    
    return sql.strip()

## 3. Traducción al Español

Las preguntas de Spider están en inglés. Hay que traducirlas al español.

In [ ]:
# Estrategias de traducción:
# 1. Usar un modelo de traducción (Helsinki-NLP/opus-mt-en-es)
# 2. Usar API de traducción (si se permite)
# 3. Reglas básicas + diccionario

def traducir_pregunta_basica(en_text):
    """Traducción básica para preguntas simples de Spider."""
    # Esta es una solución temporal - se recomienda usar un modelo de traducción
    return en_text  # placeholder

print('Se recomienda usar Helsinki-NLP/opus-mt-en-es para traducción')

## 4. Dataset BIRD

BIRD es un dataset más grande y diverso.

In [ ]:
# BIRD: https://bird-bench.github.io/
print('BIRD: https://bird-bench.github.io/')
print('Descargar y colocar en datos/bird_raw/')

## 5. Resumen

In [ ]:
print('Resumen de datos:')
print(f'  Spider: ~8,000 pares (requiere conversión)')
print(f'  BIRD: ~10,000 pares (requiere conversión)')
print(f'  Generación LLM: ~3,000+ pares (esquemas propios)')
print(f'  Objetivo: ≥2,000 pares válidos en MySQL 8.4.10')